In [6]:
# ============================
# 0️ CHECK GPU
# ============================
import torch

if torch.cuda.is_available():
    print("GPU Available:", torch.cuda.get_device_name(0))
else:
    print("GPU not found — using CPU.")


# ============================
# 1️ INSTALL DEPENDENCIES
# ============================
!pip install -q streamlit pyngrok pandas numpy scikit-learn xgboost matplotlib seaborn joblib


# ============================
# 2️ IMPORT LIBRARIES
# ============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier


# ============================
# 3️ LOAD DATASET
# ============================
file_path = '/content/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(file_path)

print(f"Loaded dataset: {df.shape}")


# ============================
# 4️ PREPROCESS DATA
# ============================

# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
).fillna(0)

# Encode target variable
df['Churn'] = df['Churn'].map({
    'Yes': 1,
    'No': 0
})

# Drop customerID
if 'customerID' in df.columns:
    df.drop('customerID', axis=1, inplace=True)


# ============================
# 5️ CREATE EDA VISUALIZATIONS
# ============================
os.makedirs('eda_images', exist_ok=True)

sns.set_style("whitegrid")

# ----------------------------
# COUNT PLOTS
# ----------------------------
categorical_features = [
    'Contract',
    'PaymentMethod',
    'InternetService',
    'TechSupport'
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(categorical_features):
    sns.countplot(
        x=col,
        hue='Churn',
        data=df,
        ax=axes[i],
        palette='viridis'
    )

    axes[i].set_title(f'Churn by {col}')
    axes[i].tick_params(axis='x', rotation=30)

plt.tight_layout()

plt.savefig(
    'eda_images/countplots.png',
    bbox_inches='tight'
)

plt.close()


# ----------------------------
# KDE PLOTS
# ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.kdeplot(
    data=df,
    x='tenure',
    hue='Churn',
    fill=True,
    ax=axes[0],
    palette='viridis'
)

axes[0].set_title('Tenure Distribution by Churn')

sns.kdeplot(
    data=df,
    x='MonthlyCharges',
    hue='Churn',
    fill=True,
    ax=axes[1],
    palette='viridis'
)

axes[1].set_title('Monthly Charges Distribution by Churn')

plt.tight_layout()

plt.savefig(
    'eda_images/kdeplots.png',
    bbox_inches='tight'
)

plt.close()


# ----------------------------
# ONE-HOT ENCODING
# ----------------------------
df_dummies = pd.get_dummies(df, drop_first=True)


# ----------------------------
# CORRELATION HEATMAP
# ----------------------------
plt.figure(figsize=(16, 12))

sns.heatmap(
    df_dummies.corr(),
    cmap='viridis'
)

plt.title('Correlation Heatmap')

plt.savefig(
    'eda_images/correlation_heatmap.png',
    bbox_inches='tight'
)

plt.close()


# ============================
# 6️ FEATURE/TARGET SPLIT
# ============================
X = df_dummies.drop('Churn', axis=1)

y = df_dummies['Churn']


# ============================
# 7️ TRAIN TEST SPLIT
# ============================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.2,
    random_state=42
)


# ============================
# 8️ FEATURE IMPORTANCE
# ============================
rf = RandomForestClassifier(
    random_state=42
)

rf.fit(X_train, y_train)

importances = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(12, 10))

sns.barplot(
    x=importances.values,
    y=importances.index,
    hue=importances.index,
    legend=False,
    palette='viridis'
)

plt.title('Feature Importance (RandomForest)')

plt.tight_layout()

plt.savefig(
    'eda_images/feature_importance.png',
    bbox_inches='tight'
)

plt.close()


# ============================
# 9️ TRAIN GPU XGBOOST MODEL
# ============================
print("Training XGBoost Model...")

if torch.cuda.is_available():
    device = "cuda"
    print("Using GPU")
else:
    device = "cpu"
    print("Using CPU")

model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='binary:logistic',
    tree_method='hist',
    device=device,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

print("Model training complete.")


# ============================
# 10️ EVALUATE MODEL
# ============================
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)

report = classification_report(
    y_test,
    y_pred,
    target_names=['No Churn', 'Churn']
)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('Confusion Matrix')

plt.savefig(
    'eda_images/confusion_matrix.png',
    bbox_inches='tight'
)

plt.close()

print(f"Model Accuracy: {acc:.2%}")

print("\nClassification Report:\n")
print(report)


# ============================
# 11️ SAVE MODEL ARTIFACTS
# ============================
joblib.dump(
    model,
    'model_xgb.pkl'
)

json.dump(
    list(X.columns),
    open('x_columns.json', 'w')
)

json.dump(
    {
        'accuracy': float(acc),
        'report': report
    },
    open('eval.json', 'w')
)

print("Artifacts saved successfully.")


# ============================
# 12️ CREATE STREAMLIT APP
# ============================
app_code = r"""
import streamlit as st
import pandas as pd
import joblib
import json
import os

# ============================
# PAGE CONFIG
# ============================
st.set_page_config(
    page_title="Telco Churn Prediction",
    layout="wide"
)

st.title("Telco Customer Churn Prediction (GPU XGBoost Model)")

# ============================
# LOAD ASSETS
# ============================
model = joblib.load('model_xgb.pkl')

columns = json.load(open('x_columns.json'))

eval_info = json.load(open('eval.json'))

EDA_DIR = 'eda_images'


# ============================
# MODEL SUMMARY
# ============================
st.header("1️ Model Evaluation Summary")

st.write(f"Accuracy: {eval_info['accuracy']:.2%}")

st.text(eval_info['report'])


# ============================
# DATASET OVERVIEW
# ============================
st.header("2️ Dataset Overview")

df = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn.csv')

st.dataframe(df.head())


# ============================
# DISPLAY EDA IMAGES
# ============================
st.header("3️ Exploratory Data Analysis")

images = [
    'countplots.png',
    'kdeplots.png',
    'correlation_heatmap.png',
    'feature_importance.png',
    'confusion_matrix.png'
]

cols = st.columns(3)

for i, img in enumerate(images):

    path = os.path.join(EDA_DIR, img)

    if os.path.exists(path):

        with cols[i % 3]:

            st.image(
                path,
                caption=img.replace('.png', '').replace('_', ' ').title(),
                use_container_width=True
            )


# ============================
# PREDICTION FORM
# ============================
st.header("4️ Predict Single Customer Churn")

with st.form("predict_form"):

    tenure = st.number_input(
        "Tenure (months)",
        0,
        200,
        12
    )

    MonthlyCharges = st.number_input(
        "Monthly Charges",
        0.0,
        10000.0,
        70.0
    )

    TotalCharges = st.number_input(
        "Total Charges",
        0.0,
        100000.0,
        3000.0
    )

    Contract = st.selectbox(
        "Contract",
        [
            "Month-to-month",
            "One year",
            "Two year"
        ]
    )

    InternetService = st.selectbox(
        "Internet Service",
        [
            "DSL",
            "Fiber optic",
            "No"
        ]
    )

    PaymentMethod = st.selectbox(
        "Payment Method",
        [
            "Electronic check",
            "Mailed check",
            "Bank transfer (automatic)",
            "Credit card (automatic)"
        ]
    )

    submitted = st.form_submit_button("Predict")


# ============================
# PREDICT
# ============================
if submitted:

    input_df = pd.DataFrame([{
        'tenure': tenure,
        'MonthlyCharges': MonthlyCharges,
        'TotalCharges': TotalCharges,
        'Contract': Contract,
        'InternetService': InternetService,
        'PaymentMethod': PaymentMethod
    }])

    input_proc = pd.get_dummies(
        input_df,
        drop_first=True
    )

    input_proc = input_proc.reindex(
        columns=columns,
        fill_value=0
    )

    pred = model.predict(input_proc)[0]

    prob = model.predict_proba(input_proc)[0][1]

    if pred == 1:

        st.error(
            f"Customer likely to CHURN — Probability: {prob:.2%}"
        )

    else:

        st.success(
            f"Customer likely to STAY — Churn Probability: {prob:.2%}"
        )
"""

with open('app.py', 'w') as f:
    f.write(app_code)

print("Streamlit app written to app.py")


# ============================
# 13️ RUN STREAMLIT + NGROK
# ============================
from pyngrok import ngrok
import subprocess
import time

# REPLACE WITH YOUR OWN NGROK TOKEN
!ngrok config add-authtoken "2nqeVqsCXdM1gwFuor0ysVSAMh7_5Xmztp16q89F4XhUe7KiE"

subprocess.Popen([
    "streamlit",
    "run",
    "app.py",
    "--server.port",
    "8501"
])

time.sleep(10)

public_url = ngrok.connect(8501)

print("\nOPEN THIS LINK:\n")
print(public_url)

GPU Available: Tesla T4
Loaded dataset: (7043, 21)
Training XGBoost Model...
Using GPU
Model training complete.
Model Accuracy: 80.34%

Classification Report:

              precision    recall  f1-score   support

    No Churn       0.84      0.90      0.87      1035
       Churn       0.66      0.54      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.80      1409

Artifacts saved successfully.
Streamlit app written to app.py
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml

OPEN THIS LINK:

NgrokTunnel: "https://be50-34-125-56-15.ngrok-free.app" -> "http://localhost:8501"
